In [ ]:
import pandas as pd
import re
import emoji


df = pd.read_csv("../data/shinline_cafe_reviews_test.csv")


emoji_pattern = re.compile(
    '['
    '\U00010000-\U0010ffff'
    '\u2600-\u27BF' 
    ']', 
    flags=re.UNICODE
)

def review_cleaner(text):
    if not isinstance(text, str):
        return ""
    text = emoji.replace_emoji(text, replace=" ")
    text = emoji_pattern.sub(" ", text)
    text = re.sub(r'([^\w\s^])\1+', r'\1', text)
    text = re.sub(r'([ㄱ-ㅎㅏ-ㅣ])\1+', r'\1', text)
    

    text = re.sub(r'\s+', ' ', text)
    return text.strip()


df['리뷰내용'] = df['리뷰내용'].apply(lambda x: review_cleaner(x))

In [36]:
df

,사업장명,place_id,리뷰내용,작성일
0,어반커피,12066484,2층에 있는 커피숍! 내부는 깨끗하고 친구들과 수다 떨기 넘 좋아요! 쿠키 프라페가...,25.12.27.
1,어반커피,12066484,내부 깨끗. 친절하시고 음료 넘넘 맛있어요~ 특히 아이스푸딩크림라떼! 가격도 괜찬코...,25.12.27.
2,어반커피,12066484,"말렌카는 맛있었는데,음료는 좀 아쉬웠어요ㅠ메뉴에 그려져있던 체리블라썸이랑은 넘 다른...",25.8.31.
3,어반커피,12066484,커피 음료 가격이 비싼편이라 그런가 한가로웠어요 인테리어 이쁘고 커피맛도 괜찮았지만...,25.7.4.
4,어반커피,12066484,진짜 최악입니다 디저트 자체 맛은 괜찮았는데 매장 관리가 아예 안되고 있는듯 자리에...,25.6.7.
...,...,...,...,...
15228,매머드 익스프레스 신림대학점,1746293546,작은 사이즈 있어서 좋아요,22.2.20.
15229,매머드 익스프레스 신림대학점,1746293546,친절하시고 커피가 저렴해서 좋아요,22.2.7.
15230,매머드 익스프레스 신림대학점,1746293546,딸기 요거트 스무디가 진짜 존맛,22.1.29.
15231,매머드 익스프레스 신림대학점,1746293546,가격이 저렴하다고 커피 맛이 안좋다는 편견을 깨주는 브랜드입니다 ㅎ,21.12.25.


In [3]:
df

,사업장명,place_id,리뷰내용,작성일
0,어반커피,12066484,2층에 있는 커피숍! 내부는 깨끗하고 친구들과 수다 떨기 넘 좋아요! 쿠키 프라페가...,25.12.27.
1,어반커피,12066484,내부 깨끗. 친절하시고 음료 넘넘 맛있어요~ 특히 아이스푸딩크림라떼! 가격도 괜찬코...,25.12.27.
2,어반커피,12066484,"말렌카는 맛있었는데,음료는 좀 아쉬웠어요ㅠ메뉴에 그려져있던 체리블라썸이랑은 넘 다른...",25.8.31.
3,어반커피,12066484,커피 음료 가격이 비싼편이라 그런가 한가로웠어요 인테리어 이쁘고 커피맛도 괜찮았지만...,25.7.4.
4,어반커피,12066484,진짜 최악입니다 디저트 자체 맛은 괜찮았는데 매장 관리가 아예 안되고 있는듯 자리에...,25.6.7.
...,...,...,...,...
15228,매머드 익스프레스 신림대학점,1746293546,작은 사이즈 있어서 좋아요,22.2.20.
15229,매머드 익스프레스 신림대학점,1746293546,친절하시고 커피가 저렴해서 좋아요,22.2.7.
15230,매머드 익스프레스 신림대학점,1746293546,딸기 요거트 스무디가 진짜 존맛,22.1.29.
15231,매머드 익스프레스 신림대학점,1746293546,가격이 저렴하다고 커피 맛이 안좋다는 편견을 깨주는 브랜드입니다 ㅎ,21.12.25.


# gliner-ko를 통한 ner tagging

In [ ]:
from gliner import GLiNER

model = GLiNER.from_pretrained("taeminlee/gliner_ko")
#model = GLiNER.from_pretrained("lots-o/gliner-bi-ko-xlarge-v1")

labels = [
    "menu", "food", "product",
    "facility", "amenity",
    "atmosphere", "vibe", "theme",
    "target audience", "companion"
]

text = "이 카페는 크로플 맛집이에요! 루프탑 뷰가 정말 예뻐서 연인과 데이트하기 좋은 로맨틱한 분위기입니다. 화장실도 깨끗해요."

entities = model.predict_entities(text, labels, threshold=0.3)

print("------GLiNER 추출 결과-----")
for entity in entities:
    word = entity["text"]
    label = entity["label"]
    score = entity["score"]
    
    category = ""
    if label in ["menu", "food", "product"]:
        category = "[MENU]"
    elif label in ["facility", "amenity"]:
        category = "[FACILITY]"
    elif label in ["atmosphere", "vibe", "theme"]:
        category = "[ATMOSPHERE]"
    elif label in ["target audience", "companion"]:
        category = "[TARGET]"
        
    print(f"개체: {word:10} | 예측 라벨: {label:15} | 카테고리: {category:12} | 확신도: {score:.2f}")

c:\VSCode\cafe_recomendation\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\VSCode\cafe_recomendation\.venv\lib\site-packages\huggingface_hub\utils\_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 3975.64it/s]


=== GLiNER 추출 결과 (영문 라벨) ===
개체: 크로플        | 예측 라벨: atmosphere      | 카테고리: [ATMOSPHERE] | 확신도: 0.69
개체: 루프탑        | 예측 라벨: menu            | 카테고리: [MENU]       | 확신도: 0.66


# 결과
- gliner 모델 중에서 한국어로 파인튜닝 된 모델 중 가장 성능이 좋은 모델임
- 그래도 결과가 아쉬움
- 단일 라벨인 "음식", "위치" 정도만 ner-tagging을 진행을 했을 땐 높은 recall 결과를 보여줌
- 하지만 prediction 부분은 매우 좋지 않음